# Trying to combine Battle class with States

## Documentation / README

The dataclasses `Player`, `Team`, and `Pokemon` seen below are fairly self-explanatory.
* The `Pokemon.base` is the "base form" of the pokemon, possibly different from `Pokemon.forme`. (Ex: `base` = `Tauros` and `forme` = `Tauros-Paldea-Aqua`)
-----
**The `Battle` class (quick facts):** (for brevity, anything of the form `.x` means `self.x`)
* `.id` (ex: `gen9randombattle-2631360263`)
* `.format` (ex: `[Gen 9] Random Battle`)
* `.p1`, `.p2` are `Player` objects, with fixed `side`s.
* `.start_time`: game start time (in seconds since the 'Epoch')
* `.winner`, `.loser`: Player objects
* `.lead_pokemon`: format `[ <lead1>, <lead2> ]` 
* `.teams`: format `[ <teamDict1>, <teamDict2> ]`

<u>Logs</u>
* `.log`: the main text log
* `.inputlog`: extra thing that contains the team seeds etc.
* `.head`: everything before '|start'
* `.bat`: everything in range `['|start', '|win|')`
* `.tail` everything after `.bat`

<u>States</u>
* `.TURNS`: Array of turn-strings from splitting `.bat`. (*Note 'turn0' = fielding leading pokemon)
    * `.TURNS[i]` gives the string for Turn `i`
* `.STATES`: List of BattleStates (incl State_0).

------
**`BattleState`:**
* `.time` : 'absolute' time turn occured at (seconds since *Epoch*)
* `.turn`
* `.team1`, `.team2`
* `.time_elapsed` : seconds passed since `Battle.start_time`
* `.battle_id` : copy of `Battle.id`; useful for printing errors.
* `.print()` gives a nice printed summary.

## Basic Classes

In [1]:
import json, copy
import re # regex
import time # parsing raw 'seconds' into readable dates

from copy import deepcopy
from dataclasses import dataclass, asdict

import logging # helpful for customizing error messages
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

# a `dataclass` is just a Class that only has attributes and no methods
# writing @dataclass lets you skip writing `__init__(...) : self.x = x` a bunch
@dataclass
class Player: 
    name: str
    side: int
    elo0: int # old
    elo1: int # new

class Team:
    def __init__(self, side: int, active: str, D: dict):
        self.side = side
        self.active = active
        self.D = D # team dictionary

    def __repr__(self):
        _out = "{" + f"side: {self.side}, active: {self.active}, D: <dict>" + "}"
        return _out

class Pokemon:
    def __init__(self, base, forme, lvl, hp, hp_max):
        self.base = base
        self.forme = forme
        self.lvl = int(lvl)
        self.hp = int(hp)
        self.hp_max = int(hp_max)

    def __repr__(self):
        _out = "{" + f"forme: {self.forme}, lvl: {self.lvl}, hp: {self.hp_max}" + "}"
        return _out

# -----------------------------
# Extra functions to help with printing teams

def team_str(team: dict):
    _out = ""
    for poke in team.keys():
        _out += f" > {poke}: {str(team[poke])}\n"
    return _out

def team_full_str(team_full: dict):
    _out = ""
    for poke in team_full.keys() :
        poke_entry = team_full[poke]
        dict_to_show = {key : poke_entry[key] for key in ['species', 'level']}
        dict_to_show['hp'] = poke_entry['stats']['hp']
        _out += f" > {poke}: {dict_to_show}\n"
    return _out
        

## `Battle`

In [2]:
class Battle:
    def __init__(self,file_name, verbose=False):
        with open("replays/gen9-randombattle/" + file_name,"r") as battle_json:
            data = json.load(battle_json)
        # -----------------------------
        # Initializing metadata/attributes
        self.id = data['id']
        self.format = data['format']
        self.formatid = data['formatid']
        
        self.rating = data['rating']
        self.rated = (self.rating != None)
        
        # self.time_list = [] # times can be wrapped into different Turn/State classes
        self.start_time = 0
        self.end_time = 0

        self.winner = ""
        self.loser = ""

        self.player_dets = data['player_dets'] # more info about players... may be redudant with self.p1, self.p2
        self.teams_full = data['teams_full'] # full teams including stats

        
        # -----------------------------
        # log setup
        self.inputlog = data.get("inputlog", "")
        self.log = data["log"]
        self.log = re.sub(r'\n\|\n', '\n', self.log) # delete any lines that are only `|`

        # `head` takes 'START'->'|start', `tail` takes '|win|'->'END', and `bat` is what's in-between.
        self.head, self.bat, self.tail = self._head_sep(self.log)

        # -----------------------------
        # processing `head`
        self.gametype = self._init_gametype(self.head)
        self.custom_ruleQ = (re.search(r'custom rule', self.head) != None)
        
        self.start_time = self._init_time(self.head)
        
        # wrapped player_names[] etc. into Player objects
        try:
            self.p1, self.p2 = self._init_players(self.head)
        except:
            logger.error(f"error in parsing `head` of battle {self.id}")

        # -----------------------------
        # processing `bat`
        self.bat = re.sub(r'\|start\n', '|turn|0\n', self.bat)
        self.TURNS = re.split(r'\|turn\|', self.bat)[1:] # discard initial ''
        
    # initialize/parse starting state ("turn 0")
        
        # Need to allow for Zoroark weirdness:
        if 'Zoroark' in self.teams_full[0].keys() :
            poke = self.teams_full[0]['Zoroark']
            D0_0 = {poke['name'] : Pokemon(poke['name'], poke['species'], poke['level'], poke['stats']['hp'], poke['stats']['hp'])}
        else : D0_0 = {}
        if 'Zoroark' in self.teams_full[1].keys() :
            poke = self.teams_full[1]['Zoroark']
            D1_0 = {poke['name'] : Pokemon(poke['name'], poke['species'], poke['level'], poke['stats']['hp'], poke['stats']['hp'])}
        else : D1_0 = {}
        
        BS_0 = BattleState(
            Team(side=1, active='', D=D0_0),
            Team(side=2, active='', D=D1_0), 
            self.TURNS[0],
            battle_id = self.id
        )
        BS_0.time = self.start_time # turn 0 time is match start
        
        self.STATES = [BS_0]
        for i in range(len(self.TURNS)-1): # -1 b/c I use `i+1` below
            BS_i = self.STATES[i]
            BS_new = BattleState(
                copy.deepcopy(BS_i.team1), # can't leave out the deepcopy!
                copy.deepcopy(BS_i.team2), # can't leave out the deepcopy!
                self.TURNS[i+1],
                match_start_time = self.start_time,
                battle_id = self.id
            )
            self.STATES.append(BS_new)
            if verbose : 
                BS_new.print()
        
        self.lead_pokemon = [
            BS_0.team1.active, 
            BS_0.team2.active
        ] # [<starter>]

        # -----------------------------
        # processing `tail`
        self.parse_tail(self.tail)
        
        self.teams = [
            self.clean_team(self.STATES[-1].team1.D), 
            self.clean_team(self.STATES[-1].team2.D)
        ] # [<{team_set}>]

    # =================================
    # END __init__()



    
    # =================================
    def __repr__(self):
        to_return = f"%%%%%%%%%%   Battle {self.id}   %%%%%%%%%%\n"; 
        to_return += f"============================================================\n";
        
        if self.rated:
            to_return += f"This was a battle between {self.p1.name} (pre-match rating {self.p1.elo0}) "
            to_return += f"and {self.p2.name} (pre-match rating {self.p2.elo0}).\n\n"
        else:
            to_return += f"This was a battle between {self.p1.name} and {self.p2.name}.\n\n"
        
        to_return += f"{self.p1.name}'s lead pokemon was {self.lead_pokemon[0]}, "
        to_return += f"and their team (by `base`) was\n{team_full_str(self.teams_full[0])}\n"
        to_return += f"{self.p2.name}'s lead pokemon was {self.lead_pokemon[1]}, "
        to_return += f"and their team (by `base`) was\n{team_full_str(self.teams_full[1])}\n"
        
        to_return += f"{self.winner.name} won!\n"
        if self.rated:
            to_return += f"{self.winner.name}'s rating increased to {self.winner.elo1}.\n"
            to_return += f"{self.loser.name}'s rating fell to {self.loser.elo1}.\n"
        else:
            to_return += "This was an unrated match, so no one's rating changed.\n"
        
        to_return += f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%\n\n"
        return to_return

        
    # These are meant to be run only in __init__
    # =================================
    def _head_sep(self, log): # separate log into 'header' and 'battle'
        head_end = log.index('|start\n')
        bat_end = log.index('\n|win|') + 1 # in case string `|win|` appears elsewhere
        return log[:head_end], log[head_end:bat_end], log[bat_end:]

    def _init_time(self, head):
        secs = re.search(r'\|t:\|(\d+)$', head, re.M).group(1)
        return int(secs) # returns a `time.tm` object
    
    def _init_gametype(self, head):
        return re.search(r'\|gametype\|(\w*)$', head, re.M).group(1)
    
    def _init_players(self, head):
        # test: |player|p1|kaisarian|lucas-gen4pt|
        # test: |player|p2|Flamesenpai557|101|
        player_pat = re.compile(r'\|player\|p(?P<side>\d)\|(?P<name>.*)\|(.*)\|(?P<elo>\d+)?\n'); # player line pattern

        p1 = player_pat.search(head)
        search_ind = p1.end() # advance the search beginning
        p2 = player_pat.search(head, pos=search_ind)

        # if battle unrated then elo is 'None'; we set to 0
        if p1.group('elo') == None : p1_elo = 0
        else : p1_elo = p1.group('elo') 
        
        if p2.group('elo') == None : p2_elo = 0
        else : p2_elo = p2.group('elo')
            
        player1 = Player(
            name = p1.group('name'), 
            side = int(p1.group('side')),
            elo0 = int(p1_elo),
            elo1 = int(p1_elo), # set equal for now
        )
        player2 = Player(
            name = p2.group('name'), 
            side = int(p2.group('side')),
            elo0 = int(p2_elo),
            elo1 = int(p2_elo), # set equal for now
        )
        
        return player1, player2

        
    # =================================
    def parse_tail(self, tail):
        match = re.match(r'\|win\|(.*)?\n', tail)

        try:
            win_name = match.group(1)
        except AttributeError as e:
            print("Error parsing |win| line.")
            return None
        
        if win_name == self.p1.name : 
            self.winner = self.p1
        else: 
            self.winner = self.p2

        if self.winner.side == 1 : 
            self.loser = self.p2
        else:
            self.loser = self.p1

        # parse Elo changes by finding deltas +/-, then applying
        if self.rated : 
            match_lose = re.search(r'\|raw\|(?:.*?)\([-+](?P<EloLoss>\d+) for losing\)\n', tail)
            match_win = re.search(r'\|raw\|(?:.*?)\([-\+](?P<EloGain>\d+) for winning\)\n', tail)
    
            try:
                elo_loss = int(match_lose.group('EloLoss'))
                self.loser.elo1 -= elo_loss # detract points
            except AttributeError as e:
                print("Error parsing Elo for loser "+f"(id:{self.id})")
            try: 
                elo_gain = int(match_win.group('EloGain'))
                self.winner.elo1 += elo_gain
            except AttributeError as e:
                print("Error parsing Elo for winner "+f"(id:{self.id})")

        return None

    
    # =================================
    # in essence this just resets all team HP's to max for printing purposes
    def clean_team(self, teamD: dict):
        if teamD == {} : 
            print("Need to pass a nonempty Team object.")
            return None
            
        _team = copy.deepcopy(teamD)
        for form in _team.keys():
            _team[form].hp = _team[form].hp_max
        return _team
        

## `BattleState`

In [3]:
class BattleState:
    # feed current teams and turn string, plus optional battle start_time and id (for debugging)
    def __init__(self, team1, team2, turn: str, match_start_time = 0, battle_id = ''): 
        self.turn = int(re.match(r'(\d+)\n', turn).group(1)) # split turn-strings start with `#\n`
        
        # update self.time if a timestamp appears in `turn`
        self.time = 0
        t_match = re.search(r'\|t:\|(\d+)\n', turn)
        if t_match != None :
            self.time = int(t_match.group(1))
        
        self.team1 = team1 
        self.team2 = team2 
        
        self.elapsed_time = self.time - match_start_time
        self.battle_id = battle_id
        
        self.parse_turn(turn) # main builder

    # =================================
    def __repr__(self):
        return f"<BattleState;turn{self.turn}>"

    # =================================
    # Separating this from __repr__ makes looking at arrays of BattleStates easier
    def print(self):
        _out = ""
        _out += f"State at END of Turn {self.turn}: \n"
        _out += f"  Match time: {time.strftime("%M:%S",time.gmtime(self.elapsed_time))} (mm:ss)\n"
        _out += f"  Team: {self.team1}\n"
        _out += f"  Team: {self.team2}\n"
        print(_out)

    
    # =================================
    # Line Parsers 
    # =================================
    # These accept only single lines
    
    def parse_switch(self, s: str):
        # test string: '|switch|p1a: Delphox|Delphox, L84, F|263/263'
        match = re.match(r"\|switch\|p(?P<plr>\d)a?: (?P<base>[\w'\- ]+)\|(?P<forme>[\w'\- ]+), (?P<lvl>L\d+)?(?:.*?)\|(?P<hp>\d+)/(?P<hpmax>\d+)(?:.*?)$", s, re.M)
        D = match.groupdict() # dictionary of captured 'groups'; for brevity
        
        player = int(D['plr'])
        base = D['base']
        forme = D['forme']
        hp = int(D['hp'])
        hpmax = int(D['hpmax'])

        # level 100 pokemon do not have a listed `level` in the log
        if D['lvl'] == None : lvl = 100
        else : lvl = int(D['lvl'][1:])

        if player == 1:
            poke = Pokemon(base, forme, lvl, hp, hpmax)
            self.team1.D[base] = poke
            self.team1.active = forme
        else: 
            poke = Pokemon(base, forme, lvl, hp, hpmax)
            self.team2.D[base] = poke
            self.team2.active = forme
        return None

    # =================================
    def parse_drag(self, s: str):
        # test string: '|drag|p1a: Delphox|Delphox, L84, F|263/263'
        match = re.match(r"\|drag\|p(?P<plr>\d)a?: (?P<base>[\w\- ]+)\|(?P<forme>[\w'\- ]+), (?P<lvl>L\d+)?(?:.*?)\|(?P<hp>\d+)/(?P<hpmax>\d+)(?:.*?)$", s, re.M)
        D = match.groupdict() # dictionary of captured 'groups'; for brevity
        
        player = int(D['plr'])
        base = D['base']
        forme = D['forme']
        hp = int(D['hp'])
        hpmax = int(D['hpmax'])

        # level 100 pokemon do not have a listed `level` in the log
        if D['lvl'] == None : lvl = 100
        else : lvl = int(D['lvl'][1:]) # [1:] to map 'L##' |-> '##'
        
        if player == 1:
            poke = Pokemon(base, forme, lvl, hp, hpmax)
            self.team1.D[base] = poke
            self.team1.active = forme
        else: 
            poke = Pokemon(base, forme, lvl, hp, hpmax)
            self.team2.D[base] = poke
            self.team2.active = forme
        return None
    
    # =================================
    def parse_damage(self, s: str):
        # test strings
        # S1 = '|-damage|p2a: Snorlax|291/397|[from] item: Rocky Helmet|[of] p1a: Skarmory'
        # S2 = '|-damage|p1a: Wugtrio|0 fnt'
            
        # most damage lines look like this
        match = re.match(r"\|-damage\|p(?P<plr>\d)a?: (?P<base>[\w'\- ]+)\|(?P<hp>\d+)/(?P<hpmax>\d+).*$", s, re.M)
        if match == None :
            # could be a 'fainting line'
            match = re.match(r"\|-damage\|p(?P<plr>\d)a?: (?P<base>[\w'\- ]+)\|(?P<hp>\d+) fnt(?:.*?)$", s, re.M)
    
        # if neither work, stop
        if match == None : 
            print("Could not parse line:\n%s" % s)
            return None
            
        D = match.groupdict() # for brevity
        
        player = int(D['plr'])
        base = D['base']
        hp = int(D['hp'])
        
        if player == 1 : 
            poke = self.team1.D[base]
            poke.hp = hp
            if D.get('hpmax') != None : poke.hp_max = int(D['hpmax'])
        else:
            poke = self.team2.D[base]
            poke.hp = hp
            if D.get('hpmax') != None : poke.hp_max = int(D['hpmax'])
        return None

    # =================================
    def parse_heal(self, s: str):
        # test strings
        # S1 = '|-heal|p1a: Skarmory|169/235'
        # S2 = '|-heal|p2a: Wigglytuff|423/424|[from] item: Leftovers'
        
        match = re.match(r"\|-heal\|p(?P<plr>\d)a?:\s(?P<base>[\w'\- ]+)\|(?P<hp>\d+)/(?P<hpmax>\d+).*$", s, re.M)
        D = match.groupdict() # for brevity
        
        player = int(D['plr'])
        base = D['base']
        hp = D['hp']
        
        if player == 1 : 
            poke = self.team1.D[base]
            poke.hp = hp
            if D.get('hpmax') != None : poke.hp_max = int(D['hpmax'])
        else:
            poke = self.team2.D[base]
            poke.hp = hp
            if D.get('hpmax') != None : poke.hp_max = int(D['hpmax'])
        return None

    # =================================
    # Main Parser
    # =================================
    def parse_turn(self, turn: str):
        for line in turn.split('\n'): 
            try:
                if line.startswith('|switch|') : self.parse_switch(line)
                elif line.startswith('|drag|') : self.parse_drag(line)
                elif line.startswith('|-damage|') : self.parse_damage(line)
                elif line.startswith('|-heal|') : self.parse_heal(line)
            except:
                print("Could not parse turn %d, line: %s (id:%s)" % (self.turn, line, self.battle_id))
                continue

# Six Examples

In [4]:
# Example 1
# set `verbose` to `True` if you want to print the state at the end of each turn
BAT1 = Battle("gen9randombattle-2631360263.json", verbose=False)
BAT1 # print(BAT1) also works

%%%%%%%%%%   Battle gen9randombattle-2631360263   %%%%%%%%%%
This was a battle between LaxMD (pre-match rating 1787) and N.TdaRajada (pre-match rating 1837).

LaxMD's lead pokemon was Delphox, and their team (by `base`) was
 > Delphox: {'species': 'Delphox', 'level': 84, 'hp': 263}
 > Indeedee: {'species': 'Indeedee-F', 'level': 90, 'hp': 272}
 > Skarmory: {'species': 'Skarmory', 'level': 80, 'hp': 235}
 > Wugtrio: {'species': 'Wugtrio', 'level': 91, 'hp': 212}
 > Tauros: {'species': 'Tauros-Paldea-Aqua', 'level': 81, 'hp': 254}
 > Victreebel: {'species': 'Victreebel', 'level': 90, 'hp': 290}

N.TdaRajada's lead pokemon was Wigglytuff, and their team (by `base`) was
 > Wigglytuff: {'species': 'Wigglytuff', 'level': 96, 'hp': 424}
 > Lokix: {'species': 'Lokix', 'level': 82, 'hp': 251}
 > Mandibuzz: {'species': 'Mandibuzz', 'level': 85, 'hp': 326}
 > Snorlax: {'species': 'Snorlax', 'level': 82, 'hp': 397}
 > Torterra: {'species': 'Torterra', 'level': 78, 'hp': 276}
 > Mienshao: {'species

In [5]:
for i in range(5):
    BAT1.STATES[i].print()

State at END of Turn 0: 
  Match time: 00:00 (mm:ss)
  Team: {side: 1, active: Delphox, D: <dict>}
  Team: {side: 2, active: Wigglytuff, D: <dict>}

State at END of Turn 1: 
  Match time: 00:08 (mm:ss)
  Team: {side: 1, active: Delphox, D: <dict>}
  Team: {side: 2, active: Wigglytuff, D: <dict>}

State at END of Turn 2: 
  Match time: 00:17 (mm:ss)
  Team: {side: 1, active: Delphox, D: <dict>}
  Team: {side: 2, active: Snorlax, D: <dict>}

State at END of Turn 3: 
  Match time: 00:24 (mm:ss)
  Team: {side: 1, active: Skarmory, D: <dict>}
  Team: {side: 2, active: Snorlax, D: <dict>}

State at END of Turn 4: 
  Match time: 00:52 (mm:ss)
  Team: {side: 1, active: Skarmory, D: <dict>}
  Team: {side: 2, active: Snorlax, D: <dict>}



In [6]:
# Example 2
BAT2 = Battle("gen9randombattle-2631363920.json")
print(BAT2)

%%%%%%%%%%   Battle gen9randombattle-2631363920   %%%%%%%%%%
This was a battle between kaisarian and Flamesenpai557.

kaisarian's lead pokemon was Mimikyu, and their team (by `base`) was
 > Mimikyu: {'species': 'Mimikyu', 'level': 79, 'hp': 216}
 > Brute Bonnet: {'species': 'Brute Bonnet', 'level': 80, 'hp': 309}
 > Breloom: {'species': 'Breloom', 'level': 83, 'hp': 235}
 > Pincurchin: {'species': 'Pincurchin', 'level': 100, 'hp': 258}
 > Overqwil: {'species': 'Overqwil', 'level': 82, 'hp': 274}
 > Ambipom: {'species': 'Ambipom', 'level': 84, 'hp': 263}

Flamesenpai557's lead pokemon was Kilowattrel, and their team (by `base`) was
 > Kilowattrel: {'species': 'Kilowattrel', 'level': 83, 'hp': 252}
 > Arceus: {'species': 'Arceus-Ghost', 'level': 69, 'hp': 280}
 > Dugtrio: {'species': 'Dugtrio-Alola', 'level': 84, 'hp': 196}
 > Gurdurr: {'species': 'Gurdurr', 'level': 85, 'hp': 283}
 > Slaking: {'species': 'Slaking', 'level': 84, 'hp': 389}
 > Breloom: {'species': 'Breloom', 'level': 83, 

In [7]:
# Example 3
BAT3 = Battle("gen9randombattle-2631365384.json")
BAT3

%%%%%%%%%%   Battle gen9randombattle-2631365384   %%%%%%%%%%
This was a battle between rgrgreger (pre-match rating 1895) and Ismusicalschool (pre-match rating 1946).

rgrgreger's lead pokemon was Electrode-Hisui, and their team (by `base`) was
 > Electrode: {'species': 'Electrode-Hisui', 'level': 87, 'hp': 246}
 > Sudowoodo: {'species': 'Sudowoodo', 'level': 94, 'hp': 284}
 > Flareon: {'species': 'Flareon', 'level': 90, 'hp': 263}
 > Primarina: {'species': 'Primarina', 'level': 83, 'hp': 268}
 > Donphan: {'species': 'Donphan', 'level': 84, 'hp': 288}
 > Latias: {'species': 'Latias', 'level': 79, 'hp': 256}

Ismusicalschool's lead pokemon was Iron Treads, and their team (by `base`) was
 > Iron Treads: {'species': 'Iron Treads', 'level': 77, 'hp': 265}
 > Lugia: {'species': 'Lugia', 'level': 73, 'hp': 275}
 > Hippowdon: {'species': 'Hippowdon', 'level': 82, 'hp': 311}
 > Slowking: {'species': 'Slowking', 'level': 88, 'hp': 310}
 > Luvdisc: {'species': 'Luvdisc', 'level': 100, 'hp': 247}


In [8]:
# Example 4
BAT4 = Battle("gen9randombattle-2631511033.json")
BAT4

%%%%%%%%%%   Battle gen9randombattle-2631511033   %%%%%%%%%%
This was a battle between MMF6G and MandoWarrior.

MMF6G's lead pokemon was Solgaleo, and their team (by `base`) was
 > Solgaleo: {'species': 'Solgaleo', 'level': 74, 'hp': 325}
 > Torterra: {'species': 'Torterra', 'level': 78, 'hp': 276}
 > Conkeldurr: {'species': 'Conkeldurr', 'level': 80, 'hp': 299}
 > Rampardos: {'species': 'Rampardos', 'level': 90, 'hp': 321}
 > Gyarados: {'species': 'Gyarados', 'level': 79, 'hp': 280}
 > Drifblim: {'species': 'Drifblim', 'level': 86, 'hp': 398}

MandoWarrior's lead pokemon was Dondozo, and their team (by `base`) was
 > Dondozo: {'species': 'Dondozo', 'level': 78, 'hp': 362}
 > Tinkaton: {'species': 'Tinkaton', 'level': 82, 'hp': 274}
 > Smeargle: {'species': 'Smeargle', 'level': 95, 'hp': 258}
 > Golduck: {'species': 'Golduck', 'level': 90, 'hp': 290}
 > Whimsicott: {'species': 'Whimsicott', 'level': 84, 'hp': 238}
 > Ting-Lu: {'species': 'Ting-Lu', 'level': 78, 'hp': 370}

MandoWarrior

In [9]:
# Example 5
BAT5 = Battle("gen9randombattle-2631533192.json")
BAT5

%%%%%%%%%%   Battle gen9randombattle-2631533192   %%%%%%%%%%
This was a battle between lalo's pollos (pre-match rating 1897) and WhadayatalkinAbout (pre-match rating 1921).

lalo's pollos's lead pokemon was Mienshao, and their team (by `base`) was
 > Mienshao: {'species': 'Mienshao', 'level': 83, 'hp': 243}
 > Regirock: {'species': 'Regirock', 'level': 83, 'hp': 268}
 > Araquanid: {'species': 'Araquanid', 'level': 82, 'hp': 246}
 > Fezandipiti: {'species': 'Fezandipiti', 'level': 82, 'hp': 278}
 > Stonjourner: {'species': 'Stonjourner', 'level': 91, 'hp': 330}
 > Espathra: {'species': 'Espathra', 'level': 80, 'hp': 283}

WhadayatalkinAbout's lead pokemon was Copperajah, and their team (by `base`) was
 > Copperajah: {'species': 'Copperajah', 'level': 86, 'hp': 350}
 > Glimmora: {'species': 'Glimmora', 'level': 75, 'hp': 248}
 > Koraidon: {'species': 'Koraidon', 'level': 64, 'hp': 235}
 > Luvdisc: {'species': 'Luvdisc', 'level': 100, 'hp': 247}
 > Ursaluna: {'species': 'Ursaluna', 'level

In [10]:
# Example 6
BAT6 = Battle("gen9randombattle-2631579047.json")
print(BAT6)

%%%%%%%%%%   Battle gen9randombattle-2631579047   %%%%%%%%%%
This was a battle between Oginotakashi and jetpackuser.

Oginotakashi's lead pokemon was Keldeo-Resolute, and their team (by `base`) was
 > Keldeo: {'species': 'Keldeo-Resolute', 'level': 79, 'hp': 273}
 > Glaceon: {'species': 'Glaceon', 'level': 94, 'hp': 275}
 > Espeon: {'species': 'Espeon', 'level': 84, 'hp': 246}
 > Dragalge: {'species': 'Dragalge', 'level': 88, 'hp': 258}
 > Glastrier: {'species': 'Glastrier', 'level': 86, 'hp': 312}
 > Kingdra: {'species': 'Kingdra', 'level': 84, 'hp': 263}

jetpackuser's lead pokemon was Salamence, and their team (by `base`) was
 > Salamence: {'species': 'Salamence', 'level': 77, 'hp': 273}
 > Armarouge: {'species': 'Armarouge', 'level': 80, 'hp': 267}
 > Donphan: {'species': 'Donphan', 'level': 84, 'hp': 288}
 > Rhyperior: {'species': 'Rhyperior', 'level': 82, 'hp': 323}
 > Arbok: {'species': 'Arbok', 'level': 87, 'hp': 246}
 > Terapagos: {'species': 'Terapagos', 'level': 77, 'hp': 26

## EDA

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [12]:
log_dir = Path("./replays/gen9-randombattle")

rows = []

stat_names = ["hp","atk","def","spa","spd","spe"]

for file in log_dir.iterdir():
    if file.is_file():
        battle = Battle(file.name)
        if not battle.custom_ruleQ:
            to_add = { # initial non-team data we may want to train on
                "id": battle.id,
                "rated": battle.rated,
                "duration": battle.end_time - battle.start_time, # end_time is currently not implemented, so this doesn't work.
                "p1": battle.p1.name,
                "p2": battle.p2.name,
                "p1_rating" : battle.p1.elo0,
                "p2_rating": battle.p2.elo0,
                "p1_wins" : battle.p1.name == battle.winner.name,
            }

            # team data and stats for each mon
            for player_index in range(2):
                for mon_index,mon_name in enumerate(battle.teams_full[player_index].keys()):
                    to_add["p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_species_id"] = battle.teams_full[player_index][mon_name]["speciesId"]
                    for stat in stat_names:
                        to_add["p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_" + stat] = battle.teams_full[player_index][mon_name]["stats"][stat]
            rows.append(to_add)

matches = pd.DataFrame(rows)

for player_index in range(2):
    for mon_index in range(6):
        new_col_name = "p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_attacking_stat"
        old_col_names = ["p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_atk",
                        "p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_spa"]
        matches[new_col_name] = matches[old_col_names].max(axis=1)
        matches = matches.drop(columns=old_col_names)

In [13]:
matches.head()

,id,rated,duration,p1,p2,p1_rating,p2_rating,p1_wins,p1_mon_1_species_id,p1_mon_1_hp,...,p1_mon_3_attacking_stat,p1_mon_4_attacking_stat,p1_mon_5_attacking_stat,p1_mon_6_attacking_stat,p2_mon_1_attacking_stat,p2_mon_2_attacking_stat,p2_mon_3_attacking_stat,p2_mon_4_attacking_stat,p2_mon_5_attacking_stat,p2_mon_6_attacking_stat
0,gen9randombattle-2631906096,True,-1781454209,sufideu,saberclaw,1135,1140,False,quaquaval,264,...,206,146,226,231,203,240,108,231,141,183
1,gen9randombattle-2631763570,False,-1781435847,PineappleCats,L4V,1959,1949,False,greninja,246,...,213,186,203,267,205,317,253,254,216,234
2,gen9randombattle-2631369343,True,-1781376557,Chicken347,cococem,1999,2068,True,munkidori,269,...,261,270,244,259,250,271,248,196,185,189
3,gen9randombattle-2631529004,False,-1781397064,WhatEver2102,Duck Cop,1999,1982,True,sinistcha,254,...,180,288,250,243,239,236,176,239,232,145
4,gen9randombattle-2631993792,False,-1781463800,monomythic,OverthereStair,2120,2062,False,ambipom,263,...,190,280,257,258,218,204,234,152,201,247


In [14]:
matches.head()

,id,rated,duration,p1,p2,p1_rating,p2_rating,p1_wins,p1_mon_1_species_id,p1_mon_1_hp,...,p1_mon_3_attacking_stat,p1_mon_4_attacking_stat,p1_mon_5_attacking_stat,p1_mon_6_attacking_stat,p2_mon_1_attacking_stat,p2_mon_2_attacking_stat,p2_mon_3_attacking_stat,p2_mon_4_attacking_stat,p2_mon_5_attacking_stat,p2_mon_6_attacking_stat
0,gen9randombattle-2631906096,True,-1781454209,sufideu,saberclaw,1135,1140,False,quaquaval,264,...,206,146,226,231,203,240,108,231,141,183
1,gen9randombattle-2631763570,False,-1781435847,PineappleCats,L4V,1959,1949,False,greninja,246,...,213,186,203,267,205,317,253,254,216,234
2,gen9randombattle-2631369343,True,-1781376557,Chicken347,cococem,1999,2068,True,munkidori,269,...,261,270,244,259,250,271,248,196,185,189
3,gen9randombattle-2631529004,False,-1781397064,WhatEver2102,Duck Cop,1999,1982,True,sinistcha,254,...,180,288,250,243,239,236,176,239,232,145
4,gen9randombattle-2631993792,False,-1781463800,monomythic,OverthereStair,2120,2062,False,ambipom,263,...,190,280,257,258,218,204,234,152,201,247


In [15]:
matches.columns

Index(['id', 'rated', 'duration', 'p1', 'p2', 'p1_rating', 'p2_rating',
       'p1_wins', 'p1_mon_1_species_id', 'p1_mon_1_hp', 'p1_mon_1_def',
       'p1_mon_1_spd', 'p1_mon_1_spe', 'p1_mon_2_species_id', 'p1_mon_2_hp',
       'p1_mon_2_def', 'p1_mon_2_spd', 'p1_mon_2_spe', 'p1_mon_3_species_id',
       'p1_mon_3_hp', 'p1_mon_3_def', 'p1_mon_3_spd', 'p1_mon_3_spe',
       'p1_mon_4_species_id', 'p1_mon_4_hp', 'p1_mon_4_def', 'p1_mon_4_spd',
       'p1_mon_4_spe', 'p1_mon_5_species_id', 'p1_mon_5_hp', 'p1_mon_5_def',
       'p1_mon_5_spd', 'p1_mon_5_spe', 'p1_mon_6_species_id', 'p1_mon_6_hp',
       'p1_mon_6_def', 'p1_mon_6_spd', 'p1_mon_6_spe', 'p2_mon_1_species_id',
       'p2_mon_1_hp', 'p2_mon_1_def', 'p2_mon_1_spd', 'p2_mon_1_spe',
       'p2_mon_2_species_id', 'p2_mon_2_hp', 'p2_mon_2_def', 'p2_mon_2_spd',
       'p2_mon_2_spe', 'p2_mon_3_species_id', 'p2_mon_3_hp', 'p2_mon_3_def',
       'p2_mon_3_spd', 'p2_mon_3_spe', 'p2_mon_4_species_id', 'p2_mon_4_hp',
       'p2_mon_4_def'

In [16]:
rated_matches = matches[matches["rated"]]

In [51]:
rated_matches.head()

,id,rated,duration,p1,p2,p1_rating,p2_rating,p1_wins,p1_mon_1_species_id,p1_mon_1_hp,...,p1_mon_3_attacking_stat,p1_mon_4_attacking_stat,p1_mon_5_attacking_stat,p1_mon_6_attacking_stat,p2_mon_1_attacking_stat,p2_mon_2_attacking_stat,p2_mon_3_attacking_stat,p2_mon_4_attacking_stat,p2_mon_5_attacking_stat,p2_mon_6_attacking_stat
0,gen9randombattle-2631906096,True,-1781454209,sufideu,saberclaw,1135,1140,False,quaquaval,264,...,206,146,226,231,203,240,108,231,141,183
2,gen9randombattle-2631369343,True,-1781376557,Chicken347,cococem,1999,2068,True,munkidori,269,...,261,270,244,259,250,271,248,196,185,189
9,gen9randombattle-2631528750,True,-1781397031,Emily Montes,Elite 4 Waally,1907,1937,True,staraptor,263,...,247,234,222,234,145,183,212,243,277,190
12,gen9randombattle-2631899911,True,-1781453538,le bolero,ear7hquak3,1901,1876,True,slowbro,300,...,207,232,253,213,207,226,203,250,236,207
13,gen9randombattle-2631506602,True,-1781393873,TRedden,MixHagelslag,1715,1712,True,excadrill,303,...,219,279,219,240,216,270,186,250,216,211


# Bevy of Tests

In [422]:
REPLAY_DIR = Path('./replays/gen9-randombattle/')
iterdir = REPLAY_DIR.iterdir()

for j in range(4000):
    replay = next(iterdir)
    if (replay.is_file() and replay.name[0]!='.'): # skipping hidden files
        try:
            _BAT = Battle(replay.name)
            
            # just for tests
            if (j % 3000 == 0) : 
                print(_BAT)
                print()
        except:
            print("Could not open file: %s" % replay.name)
            continue
    elif (replay.name[0]=='.') :
        continue
    else:
        print("Could not open file: %s" % replay.name)
        continue

%%%%%%%%%%   Battle gen9randombattle-2631906096   %%%%%%%%%%
This was a battle between sufideu (pre-match rating 1135) and saberclaw (pre-match rating 1140).

sufideu's lead pokemon was Quaquaval, and their team (by `base`) was
 > Quaquaval: {'species': 'Quaquaval', 'level': 79, 'hp': 264}
 > Pecharunt: {'species': 'Pecharunt', 'level': 77, 'hp': 262}
 > Noivern: {'species': 'Noivern', 'level': 82, 'hp': 274}
 > Azumarill: {'species': 'Azumarill', 'level': 82, 'hp': 298}
 > Gogoat: {'species': 'Gogoat', 'level': 88, 'hp': 360}
 > Klawf: {'species': 'Klawf', 'level': 90, 'hp': 272}

saberclaw's lead pokemon was Abomasnow, and their team (by `base`) was
 > Abomasnow: {'species': 'Abomasnow', 'level': 84, 'hp': 287}
 > Ceruledge: {'species': 'Ceruledge', 'level': 78, 'hp': 245}
 > Chansey: {'species': 'Chansey', 'level': 85, 'hp': 564}
 > Hippowdon: {'species': 'Hippowdon', 'level': 82, 'hp': 311}
 > Carbink: {'species': 'Carbink', 'level': 90, 'hp': 236}
 > Klefki: {'species': 'Klefki', 

# Tests

In [70]:
from pathlib import Path
REPLAYDIR = Path('./replays/gen9-randombattle')
with open(REPLAYDIR/"gen9randombattle-2631363920.json", 'r') as file:
    bat2raw = json.load(file)

In [84]:
player_pat = re.compile(r'\|player\|p(?P<side>\d)\|(?P<name>.*)\|(?P<avatar>.*?)\|(?P<elo>\d+)?\n'); # player line pattern
p1 = player_pat.search("|player|p1|kaisarian|lucas-gen4pt|\n")
p1.groupdict().get('elo',0) == None

True

# The (Original) Battle Class
This class is for making a Battle object.  A battle object takes a file name (formatted as a string, assuming a file with that name exists in data/replays/gen9-randombattle) as input.  From that string, it will gather the information of usernames, pre-battle ratings, post-battle ratings (if the match was rated), pokemon on each team, battle start and end time, and winner.  It has a nice string representation.

Next goal: figure out how to take the pokemon names and access information like type and stats

In [1]:
import json

class Battle:
    def __init__(self,file_name):
        with open("replays/gen9-randombattle/" + file_name,"r") as battle_json:
            data = json.load(battle_json)
        self.log = data["log"]
        self.player_names = [data["players"][0],data["players"][1]]
        self.old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
        self.new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
        self.winner = ""
        self.loser = ""
        self.lead_pokemon = {self.player_names[0] : "", self.player_names[1] : ""} # dictionary where keys are player usernames and values are the first pokemon each player sent onto the field
        self.teams = {self.player_names[0] : set(), self.player_names[1]: set()} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
        self.time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
        self.start_time = 0
        self.end_time = 0
        self.rated = (data["rating"] != None)
        self.log_parser(self.log)

    def __repr__(self):
        to_return = ""
        if self.rated:
            to_return += f"This was a battle between {self.player_names[0]} (pre-match rating {self.old_player_ratings[self.player_names[0]]}) and {self.player_names[1]} (pre-match rating {self.old_player_ratings[self.player_names[1]]}).\n"
        else:
            to_return += f"This was a battle between {self.player_names[0]} and {self.player_names[1]}.\n"
        for index in range(2):
            to_return += f"{self.player_names[index]}'s team was {self.teams[self.player_names[index]]} and their lead pokemon was {self.lead_pokemon[self.player_names[index]]}.\n"
        to_return += f"{self.winner} won!\n"
        if self.rated:
            to_return += f"{self.winner}'s rating increased to {self.new_player_ratings[self.winner]}.\n"
            to_return += f"{self.loser}'s rating fell to {self.new_player_ratings[self.loser]}."
        else:
            to_return += "This was an unrated match, so no one's rating changed."
        return to_return

    def log_parser(self,log):
        # This loop generates battle information by searching for *all* instances of "|t:|", "|switch|", "|win|", and " for winning)"
        # In particular, it sets the Battle object's new_player_ratings, winner, loser, teams, time_list, and rated attributes.
        for i in range(len(log)):
            # generates time list
            if log.startswith("|t:|",i):
                j = i + 4
                time_string = ""
                while log[j] != "\n":
                    time_string += log[j]
                    j += 1
                self.time_list.append(int(time_string))
            # generate team list
            if log.startswith("|switch|",i):
                pokemon_name = ""
                # Skip to the third pipe in the line.  This is where the species (including forme) is.
                pipe_counter = 0
                j = i
                while pipe_counter < 3:
                    if log[j] == "|":
                        pipe_counter += 1
                    j += 1
                # Now j is the first index after the third pipe in the line
                while log[j] != ",":
                    pokemon_name += log[j]
                    j += 1
                if log.startswith("p1a",i+8):
                    player_name = self.player_names[0]
                else:
                    player_name = self.player_names[1]
                self.teams[player_name].add(pokemon_name)
                if self.lead_pokemon[player_name] == "": # identifies  
                    self.lead_pokemon[player_name] = pokemon_name
            # identifies winner and loser
            if log.startswith("|win|",i):
                j = i + 5
                while log[j] != "\n":
                    self.winner += log[j]
                    j += 1
                winner_index = self.player_names.index(self.winner)
                self.loser = self.player_names[1-winner_index]
            if self.rated:
                if log.startswith(" for losing)",i):
                    delta_index = i-1
                    score_delta_string = ""
                    while not log[delta_index] in ["-","+"]: # if a player's rating changes by 0 after losing, the log will write +0 instead of -0 for the loser
                        score_delta_string = log[delta_index] + score_delta_string
                        delta_index -= 1
                    score_delta = int(score_delta_string)
                    new_score_index = delta_index - 17
                    new_score_string = ""
                    while log[new_score_index] != ">":
                        new_score_string = log[new_score_index] + new_score_string
                        new_score_index -= 1
                    new_score = int(new_score_string)
                    old_score = new_score + score_delta
                    self.old_player_ratings[self.loser] = old_score
                    self.new_player_ratings[self.loser] = new_score
                if log.startswith(" for winning)",i):
                    delta_index = i-1
                    score_delta_string = ""
                    while not log[delta_index] in ["+","-"]:
                        score_delta_string = log[delta_index] + score_delta_string
                        delta_index -= 1
                    score_delta = int(score_delta_string)
                    new_score_index = delta_index - 17
                    new_score_string = ""
                    while log[new_score_index] != ">":
                        new_score_string = log[new_score_index] + new_score_string
                        new_score_index -= 1
                    new_score = int(new_score_string)
                    old_score = new_score - score_delta
                    self.old_player_ratings[self.winner] = old_score
                    self.new_player_ratings[self.winner] = new_score
        # We separate start and end time for ease of access
        self.start_time = self.time_list[0]
        self.end_time = self.time_list[-1]

## Examples
Here, you can see how the Battle class functions on a couple of the files in data/replays/gen9-randombattle.  It's pretty straightforward.

In [2]:
battle1 = Battle("gen9randombattle-2631360263.json")
battle1

This was a battle between LaxMD (pre-match rating 1789) and N.TdaRajada (pre-match rating 1837).
LaxMD's team was {'Delphox', 'Tauros-Paldea-Aqua', 'Wugtrio', 'Victreebel', 'Indeedee-F', 'Skarmory'} and their lead pokemon was Delphox.
N.TdaRajada's team was {'Torterra', 'Snorlax', 'Wigglytuff', 'Lokix', 'Mandibuzz', 'Mienshao'} and their lead pokemon was Wigglytuff.
N.TdaRajada won!
N.TdaRajada's rating increased to 1854.
LaxMD's rating fell to 1772.

In [3]:
battle2 = Battle("gen9randombattle-2631363920.json")
battle2

This was a battle between kaisarian and Flamesenpai557.
kaisarian's team was {'Pincurchin', 'Ambipom', 'Mimikyu', 'Breloom', 'Overqwil', 'Brute Bonnet'} and their lead pokemon was Mimikyu.
Flamesenpai557's team was {'Arceus-Ghost', 'Kilowattrel', 'Gurdurr', 'Dugtrio-Alola', 'Slaking', 'Breloom'} and their lead pokemon was Kilowattrel.
kaisarian won!
This was an unrated match, so no one's rating changed.

In [4]:
battle3 = Battle("gen9randombattle-2631365384.json")
battle3

This was a battle between rgrgreger (pre-match rating 1895) and Ismusicalschool (pre-match rating 1946).
rgrgreger's team was {'Primarina', 'Flareon', 'Sudowoodo', 'Electrode-Hisui', 'Donphan'} and their lead pokemon was Electrode-Hisui.
Ismusicalschool's team was {'Houndoom', 'Slowking', 'Lugia', 'Hippowdon', 'Iron Treads', 'Luvdisc'} and their lead pokemon was Iron Treads.
rgrgreger won!
rgrgreger's rating increased to 1918.
Ismusicalschool's rating fell to 1923.

In [5]:
battle4 = Battle("gen9randombattle-2631511033.json")
battle4

This was a battle between MMF6G and MandoWarrior.
MMF6G's team was {'Gyarados', 'Torterra', 'Conkeldurr', 'Solgaleo', 'Rampardos', 'Drifblim'} and their lead pokemon was Solgaleo.
MandoWarrior's team was {'Whimsicott', 'Ting-Lu', 'Smeargle', 'Golduck', 'Tinkaton', 'Dondozo'} and their lead pokemon was Dondozo.
MandoWarrior won!
This was an unrated match, so no one's rating changed.

In [6]:
battle5 = Battle("gen9randombattle-2631533192.json")
battle5

This was a battle between lalo's pollos (pre-match rating 1897) and WhadayatalkinAbout (pre-match rating 1921).
lalo's pollos's team was {'Espathra', 'Stonjourner', 'Fezandipiti', 'Regirock', 'Mienshao', 'Araquanid'} and their lead pokemon was Mienshao.
WhadayatalkinAbout's team was {'Magearna', 'Koraidon', 'Glimmora', 'Copperajah'} and their lead pokemon was Copperajah.
WhadayatalkinAbout won!
WhadayatalkinAbout's rating increased to 1940.
lalo's pollos's rating fell to 1878.

# Sandbox

Below is what I was using to test.  You should probably ignore this stuff.

In [6]:
with open("replays/gen9-randombattle/gen9randombattle-2631360263.json","r") as battle_json:
    data = json.load(battle_json)
log = data["log"]
player_names = [data["players"][0],data["players"][1]]
old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
winner = ""
loser = ""
teams = {player_names[0]: set(), player_names[1]: set()} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
start_time = 0
end_time = 0

for i in range(len(log)):
    # generates time list
    if log.startswith("|t:",i):
        j = i + 4
        time_string = ""
        while log[j] != "\n":
            time_string += log[j]
            j += 1
        time_list.append(int(time_string))
    # generate team list
    if log.startswith("|switch|",i):
        pokemon_name = ""
        # Skip to the third pipe in the line.  This is where the species (including forme) is.
        pipe_counter = 0
        j = i
        while pipe_counter < 3:
            if log[j] == "|":
                pipe_counter += 1
            j += 1
        # Now j is the first index after the third pipe in the line
        while log[j] != ",":
            pokemon_name += log[j]
            j += 1
        if log.startswith("p1a",i+8):
            player_name = player_names[0]
        else:
            player_name = player_names[1]
        teams[player_name].add(pokemon_name)
    # identifies winner and loser
    if log.startswith("|win|",i):
        j = i + 5
        while log[j] != "\n":
            winner += log[j]
            j += 1
        winner_index = player_names.index(winner)
        loser = player_names[1-winner_index]
    for player_index in range(2):
        if log.startswith(f"{player_names[player_index]}'s rating: ",i):
            j = len(player_names[player_index]) + 11
            old_score_string = ""
            while log[i + j] != " ":
                old_score_string += log[i+j]
                j += 1
            old_player_ratings[player_names[player_index]] = int(old_score_string)
            new_player_ratings[player_names[player_index]] = int(log[i + j + 16:i + j + 16 + len(old_score_string)])
    

start_time = time_list[0]
end_time = time_list[-1]

for index in range(2):
    print(f"{player_names[index]}'s old rating:", old_player_ratings[player_names[index]])
    print(f"{player_names[index]}'s team:", teams[player_names[index]])
print("winner:", winner)
try:
    for index in range(2):
        print(f"{player_names[index]} new rating:", new_player_ratings[player_names[index]])
except KeyError:
    print("This was not a rated match")

print("start time:", start_time)
print("end time:", end_time)

LaxMD's old rating: 1789
LaxMD's team: {'Delphox', 'Indeedee-F', 'Skarmory', 'Victreebel', 'Tauros-Paldea-Aqua', 'Wugtrio'}
N.TdaRajada's old rating: 1837
N.TdaRajada's team: {'Wigglytuff', 'Torterra', 'Snorlax', 'Mienshao', 'Mandibuzz', 'Lokix'}
winner: N.TdaRajada
LaxMD new rating: 1772
N.TdaRajada new rating: 1854
start time: 1781375540
end time: 1781376361


In [164]:
player_name = "LaxMD"
i = log.index(f"{player_name}'s rating: ")
j = len(player_name) + 11
log[i + j]


'1'

In [43]:
len(line)

26

In [136]:
with open("replays/gen9-randombattle/gen9randombattle-2631363920.json","r") as battle_json:
    data = json.load(battle_json)

data["players"][1]

'Flamesenpai557'

In [139]:
log="|j|☆xValk_\n|j|☆manwhocouldntcook\n|t:|1781540286\n|gametype|singles\n|player|p1|xValk_|ltsurge|2138\n|player|p2|manwhocouldntcook|valerie|2146\n|gen|9\n|tier|[Gen 9] Random Battle\n|rated|\n|rule|Species Clause: Limit one of each Pokémon\n|rule|HP Percentage Mod: HP is shown in percentages\n|rule|Sleep Clause Mod: Limit one foe put to sleep\n|rule|Illusion Level Mod: Illusion disguises the Pokémon's true level\n|\n|t:|1781540286\n|teamsize|p1|6\n|teamsize|p2|6\n|start\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244\n|turn|1\n|\n|t:|1781540296\n|switch|p2a: Incineroar|Incineroar, L82, F|290/290\n|-ability|p2a: Incineroar|Intimidate|boost\n|-unboost|p1a: Ho-Oh|atk|1\n|move|p1a: Ho-Oh|Brave Bird|p2a: Incineroar\n|-damage|p2a: Incineroar|206/290\n|-damage|p1a: Ho-Oh|240/268|[from] Recoil\n|\n|upkeep\n|turn|2\n|\n|t:|1781540301\n|switch|p1a: Carbink|Carbink, L90|236/236\n|move|p2a: Incineroar|Knock Off|p1a: Carbink\n|-resisted|p1a: Carbink\n|-damage|p1a: Carbink|205/236\n|-enditem|p1a: Carbink|Chesto Berry|[from] move: Knock Off|[of] p2a: Incineroar\n|\n|upkeep\n|turn|3\n|inactive|Battle timer is ON: inactive players will automatically lose when time's up. (requested by manwhocouldntcook)\n|\n|t:|1781540308\n|switch|p2a: Vileplume|Vileplume, L85, F|266/266\n|move|p1a: Carbink|Moonblast|p2a: Vileplume\n|-resisted|p2a: Vileplume\n|-damage|p2a: Vileplume|230/266\n|\n|-heal|p2a: Vileplume|246/266|[from] item: Leftovers\n|upkeep\n|turn|4\n|\n|t:|1781540313\n|switch|p1a: Victreebel|Victreebel, L90, M|290/290\n|move|p2a: Vileplume|Sleep Powder|p1a: Victreebel\n|-immune|p1a: Victreebel\n|\n|-heal|p2a: Vileplume|262/266|[from] item: Leftovers\n|upkeep\n|turn|5\n|\n|t:|1781540316\n|move|p1a: Victreebel|Sunny Day|p1a: Victreebel\n|-weather|SunnyDay\n|move|p2a: Vileplume|Sludge Bomb|p1a: Victreebel\n|-damage|p1a: Victreebel|175/290\n|\n|-weather|SunnyDay|[upkeep]\n|-heal|p2a: Vileplume|266/266|[from] item: Leftovers\n|upkeep\n|turn|6\n|\n|t:|1781540320\n|switch|p2a: Incineroar|Incineroar, L82, F|206/290\n|-ability|p2a: Incineroar|Intimidate|boost\n|-unboost|p1a: Victreebel|atk|1\n|move|p1a: Victreebel|Sludge Wave|p2a: Incineroar\n|-damage|p2a: Incineroar|62/290\n|-damage|p1a: Victreebel|146/290|[from] item: Life Orb\n|\n|-weather|SunnyDay|[upkeep]\n|upkeep\n|turn|7\n|\n|t:|1781540325\n|move|p1a: Victreebel|Sludge Wave|p2a: Incineroar\n|-damage|p2a: Incineroar|0 fnt\n|faint|p2a: Incineroar\n|-damage|p1a: Victreebel|117/290|[from] item: Life Orb\n|\n|-weather|SunnyDay|[upkeep]\n|upkeep\n|\n|t:|1781540335\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244\n|turn|8\n|\n|t:|1781540338\n|move|p1a: Victreebel|Weather Ball|p2a: Scrafty\n|-damage|p2a: Scrafty|121/244\n|-damage|p1a: Victreebel|88/290|[from] item: Life Orb\n|move|p2a: Scrafty|Knock Off|p1a: Victreebel\n|-damage|p1a: Victreebel|0 fnt\n|-enditem|p1a: Victreebel|Life Orb|[from] move: Knock Off|[of] p2a: Scrafty\n|faint|p1a: Victreebel\n|\n|-weather|SunnyDay|[upkeep]\n|-heal|p2a: Scrafty|136/244|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540342\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|turn|9\n|\n|t:|1781540347\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Scrafty\n|-damage|p2a: Scrafty|24/244\n|-status|p2a: Scrafty|brn\n|move|p2a: Scrafty|Rest|p2a: Scrafty\n|-status|p2a: Scrafty|slp|[from] move: Rest\n|-heal|p2a: Scrafty|244/244 slp|[silent]\n|\n|-weather|none\n|upkeep\n|turn|10\n|\n|t:|1781540353\n|switch|p2a: Jolteon|Jolteon, L84, F|246/246\n|move|p1a: Ho-Oh|Brave Bird|p2a: Jolteon\n|-resisted|p2a: Jolteon\n|-damage|p2a: Jolteon|169/246\n|-damage|p1a: Ho-Oh|243/268|[from] Recoil\n|\n|-heal|p2a: Jolteon|184/246|[from] item: Leftovers\n|upkeep\n|turn|11\n|\n|t:|1781540357\n|switch|p1a: Carbink|Carbink, L90|205/236\n|move|p2a: Jolteon|Substitute|p2a: Jolteon\n|-start|p2a: Jolteon|Substitute\n|-damage|p2a: Jolteon|123/246\n|\n|-heal|p2a: Jolteon|138/246|[from] item: Leftovers\n|upkeep\n|turn|12\n|\n|t:|1781540362\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|141/236\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-end|p2a: Jolteon|Substitute\n|\n|-heal|p2a: Jolteon|153/246|[from] item: Leftovers\n|upkeep\n|turn|13\n|\n|t:|1781540368\n|switch|p2a: Vileplume|Vileplume, L85, F|266/266\n|switch|p1a: Ho-Oh|Ho-Oh, L71|268/268\n|\n|upkeep\n|turn|14\n|\n|t:|1781540372\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|68/266\n|-status|p2a: Vileplume|brn\n|move|p2a: Vileplume|Sleep Powder|p1a: Ho-Oh\n|-status|p1a: Ho-Oh|slp|[from] move: Sleep Powder\n|\n|-heal|p2a: Vileplume|84/266 brn|[from] item: Leftovers\n|-damage|p2a: Vileplume|68/266 brn|[from] brn\n|upkeep\n|turn|15\n|\n|t:|1781540377\n|cant|p1a: Ho-Oh|slp\n|move|p2a: Vileplume|Strength Sap|p1a: Ho-Oh\n|-unboost|p1a: Ho-Oh|atk|1\n|-heal|p2a: Vileplume|266/266 brn\n|\n|-damage|p2a: Vileplume|250/266 brn|[from] brn\n|upkeep\n|turn|16\n|\n|t:|1781540380\n|switch|p2a: Scrafty|Scrafty, L83, M|244/244 slp\n|-curestatus|p1a: Ho-Oh|slp|[msg]\n|move|p1a: Ho-Oh|Brave Bird|p2a: Scrafty\n|-supereffective|p2a: Scrafty\n|-damage|p2a: Scrafty|112/244 slp\n|-damage|p1a: Ho-Oh|224/268|[from] Recoil\n|\n|-heal|p2a: Scrafty|127/244 slp|[from] item: Leftovers\n|upkeep\n|turn|17\n|\n|t:|1781540387\n|move|p1a: Ho-Oh|Earthquake|p2a: Scrafty\n|-damage|p2a: Scrafty|92/244 slp\n|cant|p2a: Scrafty|slp\n|\n|-heal|p2a: Scrafty|107/244 slp|[from] item: Leftovers\n|upkeep\n|turn|18\n|\n|t:|1781540390\n|switch|p2a: Vileplume|Vileplume, L85, F|250/266 brn\n|move|p1a: Ho-Oh|Brave Bird|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|88/266 brn\n|-damage|p1a: Ho-Oh|171/268|[from] Recoil\n|\n|-heal|p2a: Vileplume|104/266 brn|[from] item: Leftovers\n|-damage|p2a: Vileplume|88/266 brn|[from] brn\n|upkeep\n|turn|19\n|\n|t:|1781540394\n|move|p1a: Ho-Oh|Sacred Fire|p2a: Vileplume\n|-supereffective|p2a: Vileplume\n|-damage|p2a: Vileplume|0 fnt\n|faint|p2a: Vileplume\n|\n|upkeep\n|\n|t:|1781540396\n|switch|p2a: Jolteon|Jolteon, L84, F|153/246\n|turn|20\n|\n|t:|1781540399\n|switch|p1a: Carbink|Carbink, L90|141/236\n|move|p2a: Jolteon|Substitute|p2a: Jolteon\n|-start|p2a: Jolteon|Substitute\n|-damage|p2a: Jolteon|92/246\n|\n|-heal|p2a: Jolteon|107/246|[from] item: Leftovers\n|upkeep\n|turn|21\n|\n|t:|1781540401\n|move|p2a: Jolteon|Calm Mind|p2a: Jolteon\n|-boost|p2a: Jolteon|spa|1\n|-boost|p2a: Jolteon|spd|1\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-end|p2a: Jolteon|Substitute\n|\n|-heal|p2a: Jolteon|122/246|[from] item: Leftovers\n|upkeep\n|turn|22\n|\n|t:|1781540403\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|44/236\n|move|p1a: Carbink|Body Press|p2a: Jolteon\n|-damage|p2a: Jolteon|8/246\n|\n|-heal|p2a: Jolteon|23/246|[from] item: Leftovers\n|upkeep\n|turn|23\n|\n|t:|1781540406\n|move|p2a: Jolteon|Thunderbolt|p1a: Carbink\n|-damage|p1a: Carbink|0 fnt\n|faint|p1a: Carbink\n|\n|-heal|p2a: Jolteon|38/246|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540407\n|switch|p1a: Cyclizar|Cyclizar, L83, M|252/252\n|turn|24\n|\n|t:|1781540410\n|-terastallize|p2a: Jolteon|Ice\n|move|p2a: Jolteon|Tera Blast|p1a: Cyclizar|[anim] Tera Blast Ice\n|-supereffective|p1a: Cyclizar\n|-damage|p1a: Cyclizar|0 fnt\n|faint|p1a: Cyclizar\n|\n|-heal|p2a: Jolteon|53/246|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540416\n|switch|p1a: Mewtwo|Mewtwo, L72|272/272\n|-ability|p1a: Mewtwo|Unnerve\n|turn|25\n|\n|t:|1781540423\n|move|p2a: Jolteon|Tera Blast|p1a: Mewtwo|[anim] Tera Blast Ice\n|-damage|p1a: Mewtwo|127/272\n|move|p1a: Mewtwo|Psystrike|p2a: Jolteon\n|-damage|p2a: Jolteon|0 fnt\n|faint|p2a: Jolteon\n|-damage|p1a: Mewtwo|100/272|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781540429\n|switch|p2a: Staraptor|Staraptor, L79, M|263/263\n|turn|26\n|\n|t:|1781540433\n|move|p1a: Mewtwo|Psystrike|p2a: Staraptor\n|-damage|p2a: Staraptor|68/263\n|-damage|p1a: Mewtwo|73/272|[from] item: Life Orb\n|move|p2a: Staraptor|U-turn|p1a: Mewtwo\n|-supereffective|p1a: Mewtwo\n|-damage|p1a: Mewtwo|0 fnt\n|faint|p1a: Mewtwo\n|\n|t:|1781540436\n|switch|p2a: Scrafty|Scrafty, L83, M|107/244 slp|[from] U-turn\n|\n|-heal|p2a: Scrafty|122/244 slp|[from] item: Leftovers\n|upkeep\n|\n|t:|1781540440\n|switch|p1a: Ho-Oh|Ho-Oh, L71|260/268\n|turn|27\n|\n|t:|1781540448\n|move|p1a: Ho-Oh|Brave Bird|p2a: Scrafty\n|-supereffective|p2a: Scrafty\n|-damage|p2a: Scrafty|0 fnt\n|faint|p2a: Scrafty\n|-damage|p1a: Ho-Oh|220/268|[from] Recoil\n|\n|upkeep\n|\n|t:|1781540452\n|switch|p2a: Staraptor|Staraptor, L79, M|68/263\n|turn|28\n|\n|t:|1781540460\n|move|p2a: Staraptor|Brave Bird|p1a: Ho-Oh\n|-damage|p1a: Ho-Oh|0 fnt\n|faint|p1a: Ho-Oh\n|-damage|p2a: Staraptor|0 fnt|[from] Recoil\n|faint|p2a: Staraptor\n|\n|upkeep\n|\n|t:|1781540462\n|switch|p2a: Hitmonlee|Hitmonlee, L85, M|224/224\n|switch|p1a: Lugia|Lugia, L73|275/275\n|turn|29\n|\n|t:|1781540468\n|move|p1a: Lugia|Aeroblast|p2a: Hitmonlee\n|-supereffective|p2a: Hitmonlee\n|-damage|p2a: Hitmonlee|102/224\n|move|p2a: Hitmonlee|Knock Off|p1a: Lugia\n|-supereffective|p1a: Lugia\n|-damage|p1a: Lugia|208/275\n|-enditem|p1a: Lugia|Heavy-Duty Boots|[from] move: Knock Off|[of] p2a: Hitmonlee\n|\n|upkeep\n|turn|30\n|\n|t:|1781540475\n|-terastallize|p1a: Lugia|Ground\n|move|p1a: Lugia|Aeroblast|p2a: Hitmonlee\n|-supereffective|p2a: Hitmonlee\n|-damage|p2a: Hitmonlee|0 fnt\n|faint|p2a: Hitmonlee\n|\n|win|xValk_\n|raw|xValk_'s rating: 2138 &rarr; <strong>2158</strong><br />(+20 for winning)\n|raw|manwhocouldntcook's rating: 2146 &rarr; <strong>2126</strong><br />(-20 for losing)\n"

In [142]:
log.index("xValk_'s rating: ")

8974

In [147]:
for i in range(len(log)):
    if log.endswith("xValk_'s rating: ", i):
        print(i)

In [150]:
dict = {"player1": set(), "player2": set()}

In [151]:
dict

{'player1': set(), 'player2': set()}

In [152]:
dict["player1"]

set()